# Advanced Regression on Phishing Datasets
This notebook applies **Linear Regression with Forward Selection**, **Linear Regression with Backward Selection**, **Principal Component Regression (PCR)**, and **Partial Least Squares Regression (PLSR)** to evaluate model performance on phishing datasets.

In [2]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')

In [3]:
def evaluate_advanced_models(file_path, target_col, cols_to_drop):
    print(f"\n{'='*60}\nEvaluating Advanced Models on {file_path}\n{'='*60}")
    
    # 1. Load and Clean Data
    df = pd.read_csv(file_path)
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
    
    # Handle missing values safely
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
            
    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Encode target variable if it contains text
    if y.dtype == 'object' or y.dtype == 'string':
        encoder = LabelEncoder()
        y = encoder.fit_transform(y)
    
    # Identify Categorical and Continuous Features
    categorical_features = [col for col in X.columns if X[col].dtype == 'object' or X[col].nunique() < 10]
    continuous_features = [col for col in X.columns if col not in categorical_features]
    
    # 2. Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), continuous_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ])
    
    X_processed = preprocessor.fit_transform(X)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)
    
    results = []
    
    # Helper to evaluate model metrics
    def eval_model(name, model, X_tr, y_tr, X_te, y_te):
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        mse = mean_squared_error(y_te, y_pred)
        r2 = r2_score(y_te, y_pred)
        results.append({"Model": name, "MSE": mse, "R2 Score": r2})
        
    lr = LinearRegression()
    
    # 3. Forward Selection
    print("Running Forward Selection... (This may take a moment)")
    # Selecting top 10 features to keep computation times reasonable
    sfs_forward = SequentialFeatureSelector(lr, direction='forward', n_features_to_select=10, cv=3)
    sfs_forward.fit(X_train, y_train)
    X_train_fwd = sfs_forward.transform(X_train)
    X_test_fwd = sfs_forward.transform(X_test)
    eval_model("Linear Regression (Forward Selection - Top 10)", lr, X_train_fwd, y_train, X_test_fwd, y_test)
    
    # 4. Backward Selection
    print("Running Backward Selection... (This may take a moment)")
    sfs_backward = SequentialFeatureSelector(lr, direction='backward', n_features_to_select=10, cv=3)
    sfs_backward.fit(X_train, y_train)
    X_train_bwd = sfs_backward.transform(X_train)
    X_test_bwd = sfs_backward.transform(X_test)
    eval_model("Linear Regression (Backward Selection - Top 10)", lr, X_train_bwd, y_train, X_test_bwd, y_test)
    
    # 5. PCR (Principal Component Regression)
    print("Running PCR...")
    # Retaining 95% of the variance
    pcr = Pipeline([('pca', PCA(n_components=0.95)), ('lr', LinearRegression())])
    eval_model("PCR (Principal Component Regression)", pcr, X_train, y_train, X_test, y_test)
    
    # 6. PLSR (Partial Least Squares Regression)
    print("Running PLSR...")
    # Using 5 components as a standard baseline
    plsr = PLSRegression(n_components=5)
    eval_model("PLSR (Partial Least Squares)", plsr, X_train, y_train, X_test, y_test)
    
    # Display Results
    results_df = pd.DataFrame(results)
    print("\nResults:\n")
    print(results_df.to_string(index=False))
    print("\n")

## Analysis of `StealthPhisher2025(2).csv`

In [ ]:
stealth_file = 'StealthPhisher2025(2).csv'
cols_to_drop_stealth = ['URL', 'Domain', 'TLD', 'HasTitle', 'Title']
evaluate_advanced_models(stealth_file, 'Label', cols_to_drop_stealth)


Evaluating Advanced Models on StealthPhisher2025(2).csv
Running Forward Selection... (This may take a moment)
Running Backward Selection... (This may take a moment)


## Analysis of `PhiUSIIL_Phishing_URL_Dataset(1).csv`

In [ ]:
phiusiil_file = 'PhiUSIIL_Phishing_URL_Dataset(1).csv'
cols_to_drop_phiusiil = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
evaluate_advanced_models(phiusiil_file, 'label', cols_to_drop_phiusiil)


Evaluating Advanced Models on PhiUSIIL_Phishing_URL_Dataset(1).csv
Running Forward Selection... (This may take a moment)
Running Backward Selection... (This may take a moment)
